# Importation

In [23]:
import os
import math
import copy
import json
import torch
import random
import warnings
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
#from torch.utils.data.sampler import WeightedRandomSampler
from typing import Dict, List, Tuple, Any
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.utils import weight_norm
import torch.optim as optim
import torch.optim.lr_scheduler as lr_scheduler
import tensorflow.keras as tf_keras
import tensorflow as tf
from transformers import AutoTokenizer
from transformers import DistilBertModel, DistilBertTokenizerFast


from models import SA_LSTM_Classification_Model, SelfAttention, BERTLSTMClassifier

# Chargement modèles

In [4]:
MODEL_VIDEO_PATH = "./video/best_sa_lstm_53,1.pt"
MODEL_AUDIO_PATH = "./audio/yamnet_classifier.pt"
MODEL_TEXT_PATH = "./texte/best_model.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

video_model = torch.load(MODEL_VIDEO_PATH, map_location=DEVICE, weights_only=False)
audio_model = torch.load(MODEL_AUDIO_PATH, map_location=DEVICE, weights_only=False)
state_dict = torch.load(MODEL_TEXT_PATH, map_location="cpu", weights_only=True)
tokenizer = DistilBertTokenizerFast.from_pretrained("./texte/finetuned_bert")


text_model = BERTLSTMClassifier(
    output_dim=20, hidden_dim=256, n_layers=2, dropout=0.4, bert_dir="./texte/finetuned_bert"
).to(DEVICE)

state = torch.load(MODEL_TEXT_PATH, map_location=DEVICE)
missing, unexpected = text_model.load_state_dict(state, strict=False)
if missing or unexpected:
    print("⚠️ State dict diff:", {"missing": missing, "unexpected": unexpected})


In [5]:
JSON_PATH = "./infos/train_val_videodatainfo.json"
VIDEO_FEAT_DIR = "./video/features_finetuned"
AUDIO_FEAT_DIR = "./audio/audios_embeddings"

# Chargements features

In [16]:

# === Chemins par défaut ===
JSON_PATH = globals().get("JSON_PATH", "train_val_videodatainfo.json")
VIDEO_FEAT_DIR = globals().get("VIDEO_FEAT_DIR", "features_video")
AUDIO_FEAT_DIR = globals().get("AUDIO_FEAT_DIR", "features_audio")
TEXT_EMB_TYPE = globals().get("TEXT_EMB_TYPE", "captions")  # pour clarté

# ===============================
# Dataset multimodal
# ===============================
class MultiModalDataset(Dataset):
    """
    Dataset multimodal MSR-VTT compatible avec BERT fine-tuné.
    Retourne : video_feats, audio_feats, captions (list[str]), label
    """
    def __init__(self, split: str, json_path: str = JSON_PATH,
                 video_dir: str = VIDEO_FEAT_DIR, audio_dir: str = AUDIO_FEAT_DIR):
        super().__init__()
        self.split = split
        self.json_path = json_path
        self.video_dir = video_dir
        self.audio_dir = audio_dir
        self.samples: List[Dict[str, Any]] = []
        self._index_from_json()

    def _index_from_json(self):
        if not os.path.isfile(self.json_path):
            raise FileNotFoundError(f"JSON file not found: {self.json_path}")
        with open(self.json_path, "r") as f:
            data = json.load(f)

        videos = data.get("videos", [])
        for v in videos:
            if v.get("split") != self.split:
                continue

            vid = v.get("video_id")
            label = int(v.get("category"))
            captions = v.get("captions", [])
            video_path = os.path.join(self.video_dir, f"{vid}.npy")
            audio_path = os.path.join(self.audio_dir, f"{vid}.npy")

            if not os.path.exists(video_path):
                warnings.warn(f"[{self.split}] Missing video features: {video_path}")
                continue

            self.samples.append({
                "video": video_path,
                "audio": audio_path if os.path.exists(audio_path) else None,
                "captions": captions if isinstance(captions, list) else [captions],
                "label": label,
            })

        if not self.samples:
            raise RuntimeError(f"No multimodal samples found for split='{self.split}'")

        print(f"[{self.split}] Indexed {len(self.samples)} samples across {len(set(s['label'] for s in self.samples))} classes.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        s = self.samples[idx]
        label = s["label"]

        # === Vidéo ===
        video_feats = np.load(s["video"], allow_pickle=True)
        video_feats = torch.tensor(video_feats, dtype=torch.float32)

        # === Audio ===
        if s["audio"] and os.path.exists(s["audio"]):
            audio_feats = np.load(s["audio"], allow_pickle=True)
            if audio_feats.ndim == 1:
                audio_feats = audio_feats.reshape(1, -1)
            elif audio_feats.ndim > 2:
                audio_feats = np.squeeze(audio_feats)
            audio_feats = torch.tensor(audio_feats, dtype=torch.float32)
        else:
            audio_feats = torch.zeros((1, 1024), dtype=torch.float32)

        captions = s["captions"]
        return video_feats, audio_feats, captions, label


def collate_fn_multimodal(batch):
    """
    Prépare un batch multimodal :
      - pad vidéo/audio
      - tokenize captions (BERT fine-tuné)
      - assemble pour le modèle multimodal
    """
    videos, audios, captions, labels = zip(*batch)

    # === Vidéo ===
    video_lens = [v.shape[0] if v.ndim >= 2 else 1 for v in videos]
    max_vlen = max(video_lens)
    vdim = videos[0].shape[-1]
    padded_videos = torch.zeros((len(videos), max_vlen, vdim), dtype=torch.float32)
    for i, v in enumerate(videos):
        L = v.shape[0]
        padded_videos[i, :L] = v[:L]

    # === Audio ===
    fixed_audios = []
    for a in audios:
        if not torch.is_tensor(a):
            a = torch.tensor(a, dtype=torch.float32)
        if a.ndim == 1:
            a = a.unsqueeze(0)
        elif a.ndim > 2:
            a = a.squeeze()
        fixed_audios.append(a)
    audio_feats = torch.stack([a.mean(dim=0) for a in fixed_audios])

    # === Texte ===
    MAX_CAPTIONS = 5
    batched_input_ids, batched_attention_masks = [], []

    for caps in captions:
        # Assure-toi que c’est une liste
        if not isinstance(caps, list):
            caps = [str(caps)]
        # Supprime None et chaînes vides
        caps = [str(c).strip() for c in caps if c and str(c).strip()]
        # Si aucune caption valide, mets une caption neutre
        if len(caps) == 0:
            caps = [""]
        # Limite le nombre de captions
        caps = random.sample(caps, min(len(caps), MAX_CAPTIONS))

        # Tokenisation robuste
        try:
            enc = tokenizer(
                caps,
                padding=True,
                truncation=True,
                max_length=64,
                return_tensors="pt"
            )
        except Exception as e:
            print(f"⚠️ Tokenization error for captions {caps}: {e}")
            enc = tokenizer([""], padding=True, truncation=True, max_length=64, return_tensors="pt")

        batched_input_ids.append(enc["input_ids"])
        batched_attention_masks.append(enc["attention_mask"])

    labels = torch.tensor(labels, dtype=torch.long)

    return {
        "video": padded_videos,
        "audio": audio_feats,
        "text_input_ids": batched_input_ids,        # list[Tensor]
        "text_attention_mask": batched_attention_masks,  # list[Tensor]
        "labels": labels,
    }



# ===============================
# Setup loader + sampler
# ===============================
def setup_multimodal_pipeline(
    batch_size=None,
    shuffle=False,
    num_workers=0,
    use_weighted_sampler=False,
    split=None,
):
    bs = batch_size or globals().get("BATCH_SIZE", 32)
    split = split or ("train" if shuffle else "validate")

    dataset = MultiModalDataset(split=split)
    labels = [s["label"] for s in dataset.samples]
    num_classes = max(labels) + 1

    # === Poids de classes ===
    class_counts = {c: labels.count(c) for c in set(labels)}
    class_weights = torch.ones(num_classes, dtype=torch.float32)
    for c, cnt in class_counts.items():
        class_weights[c] = 1.0 / max(cnt, 1)

    sampler = None
    if use_weighted_sampler:
        sample_weights = [class_weights[lbl].item() for lbl in labels]
        sampler = WeightedRandomSampler(torch.tensor(sample_weights), len(sample_weights))
        shuffle = False

    loader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=shuffle and sampler is None,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn_multimodal,
    )

    return loader, num_classes



# Construction Cross-modal audio+vidéo+texte

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiModalUnifier(nn.Module):
    def __init__(self, video_model, audio_model, text_model, num_classes, hidden_dim=256, dropout=0.3):
        """
        video_model, audio_model, text_model : sous-modèles pré-entraînés
        num_classes : nombre total de classes
        hidden_dim : dimension du MLP
        """
        super().__init__()
        self.video_model = video_model
        self.audio_model = audio_model
        self.text_model  = text_model
        self.num_classes = num_classes

        # Geler les sous-modèles (on peut les dégeler plus tard si souhaité)
        for m in [self.video_model, self.audio_model, self.text_model]:
            for p in m.parameters():
                p.requires_grad = False
            m.eval()

        # Taille d’entrée du MLP = concat des 3 logits
        self.input_dim = num_classes
        self.fusion_gate = nn.Sequential(
            nn.Linear(num_classes * 3, 3),
            nn.Softmax(dim=1)
        )

        self.proj_v = nn.Linear(num_classes, num_classes)
        self.proj_a = nn.Linear(num_classes, num_classes)
        self.proj_t = nn.Linear(num_classes, num_classes)

        # Tête de fusion MLP
        self.mlp = nn.Sequential(
            nn.Linear(self.input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, batch):
        """
        batch: dict venant du collate_fn_multimodal
        Retourne les logits fusionnés.
        """

        # --- 1️⃣ Vidéo ---
        video_feats = batch["video"]
        if self.training:
            video_feats = video_feats + 0.01 * torch.randn_like(video_feats)
        video_feats = video_feats.view(video_feats.size(0), 4, 1408)
        video_logits = self.video_model(video_feats)  # [B, C]

        # --- 2️⃣ Audio ---
        audio_feats = batch["audio"]
        if self.training:
            audio_feats = audio_feats + 0.01 * torch.randn_like(audio_feats)
        audio_logits = self.audio_model(audio_feats)  # [B, C]

        # --- 3️⃣ Texte ---
        input_ids_list = batch["text_input_ids"]
        attention_mask_list = batch["text_attention_mask"]

        # 👉 Appel direct au modèle texte, qui gère la moyenne
        text_logits = self.text_model(input_ids_list, attention_mask_list)
        text_logits = text_logits / 2.0  # T=2 → softmax plus étalé


        video_logits = self.proj_v(video_logits)
        audio_logits = self.proj_a(audio_logits)
        text_logits  = self.proj_t(text_logits)

        # Dans le forward :
        fused_inputs = torch.cat([video_logits, audio_logits, text_logits], dim=1)
        gates = self.fusion_gate(fused_inputs)  # [B, 3]
        fused = (
            gates[:, 0:1] * video_logits +
            gates[:, 1:2] * audio_logits +
            gates[:, 2:3] * text_logits
        )
        return self.mlp(fused)



# Entrainement

In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts


# --- MixUp (sur les logits, version soft) ---
def logits_mixup(logits, labels, alpha=0.2):
    """MixUp applied to logits directly."""
    if alpha <= 0:
        return logits, labels, labels, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = logits.size(0)
    if batch_size == 1:
        return logits, labels, labels, 1.0
    index = torch.randperm(batch_size, device=logits.device)
    mixed_logits = lam * logits + (1 - lam) * logits[index, :]
    y_a, y_b = labels, labels[index]
    return mixed_logits, y_a, y_b, lam

def mixup_data(x, y, alpha=0.3):
    """Classic MixUp on inputs"""
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


# --- Train one epoch ---
def train_one_epoch_unifier(model, dataloader, optimizer, criterion, device, mixup_alpha=0.2, grad_clip_value=5.0):
    model.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    for batch in tqdm(dataloader, desc="Training", leave=False):
        # Préparation des données
        for k, v in batch.items():
            if torch.is_tensor(v):
                batch[k] = v.to(device)

        labels = batch["labels"]
        optimizer.zero_grad()

        # --- MixUp partiel sur vidéo + audio (avant passage dans le modèle) ---
        if np.random.rand() < 0.5 and mixup_alpha > 0:  # 50 % des batches
            lam = np.random.beta(mixup_alpha, mixup_alpha)
            index = torch.randperm(batch["video"].size(0)).to(device)

            batch["video"] = lam * batch["video"] + (1 - lam) * batch["video"][index]
            batch["audio"] = lam * batch["audio"] + (1 - lam) * batch["audio"][index]

            # On garde labels mixtes
            y_a, y_b = labels, labels[index]
            mixup_active = True
        else:
            mixup_active = False

        # --- Forward ---
        logits = model(batch)

        # --- Compute loss ---
        if mixup_active:
            loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
        else:
            loss = criterion(logits, labels)

        # --- Backpropagation ---
        loss.backward()

        if grad_clip_value is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_value)

        optimizer.step()

        # --- Statistiques ---
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, 100 * total_correct / total_samples


# --- Validation ---
def evaluate_unifier(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validating", leave=False):
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch[k] = v.to(device)
            labels = batch["labels"]
            logits = model(batch)
            loss = criterion(logits, labels)
            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, 100 * total_correct / total_samples


# --- Full Pipeline ---
def run_pipeline_unifier(train_loader, val_loader, num_classes, save_path="best_unifier.pt"):
    print("\n--- Running Unifier Training Pipeline ---")
    print(f"Device: {DEVICE}")
    print(f"  LR: {LEARNING_RATE}, Weight Decay: {WEIGHT_DECAY}")
    print(f"  MixUp α: {MIXUP_ALPHA}, Dropout: {CLASSIFIER_DROPOUT}")
    print("-" * 40)
    alpha = MIXUP_ALPHA
    dropout_p = CLASSIFIER_DROPOUT
    # --- Model ---
    unifier = MultiModalUnifier(
        video_model=video_model,
        audio_model=audio_model,
        text_model=text_model,
        num_classes=num_classes,
        hidden_dim=HIDDEN_DIM,
        dropout=CLASSIFIER_DROPOUT
    ).to(DEVICE)

    # --- Loss / Optimizer / Scheduler ---
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(unifier.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    #scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=SCHEDULER_FACTOR,
    #                                           patience=SCHEDULER_PATIENCE, min_lr=MIN_LR)

    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=MIN_LR)

    best_val_acc, no_improve_epochs = 0.0, 0

    for epoch in range(NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
        bef = dropout_p
        if epoch < 20:
            mixup_alpha = 0.4
            dropout_p = 0.45
        elif epoch < 50:
            mixup_alpha = 0.2
            dropout_p = 0.35
        else:
            mixup_alpha = 0.1
            dropout_p = 0.25

        # appliquer dynamiquement :
        if dropout_p != bef:
          for layer in unifier.mlp:
              if isinstance(layer, nn.Dropout):
                  layer.p = dropout_p

        train_loss, train_acc = train_one_epoch_unifier(unifier, train_loader, optimizer, criterion, DEVICE, alpha)
        val_loss, val_acc = evaluate_unifier(unifier, val_loader, criterion, DEVICE)

        #scheduler.step(val_acc)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']

        print(f"Train Acc: {train_acc:.2f}% (Loss: {train_loss:.4f}) | "
              f"Val Acc: {val_acc:.2f}% (Loss: {val_loss:.4f}) | LR: {lr:.6f}")

        if val_acc > best_val_acc and val_acc + 15 >= train_acc:
            best_val_acc = val_acc
            torch.save(unifier, save_path)
            print(f"   ✅ New best val acc: {val_acc:.2f}% → Saved to {save_path}")
            no_improve_epochs = 0
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= EARLY_STOP_PATIENCE:
                print(f"--- Early stopping after {epoch+1} epochs (no improvement for {EARLY_STOP_PATIENCE}). ---")
                break

    print(f"\n--- Training Complete. Best Val Acc: {best_val_acc:.2f}% ---")
    return unifier


In [19]:

# Classes
NUM_CLASSES = 20

# Batch & epochs
BATCH_SIZE = 256
NUM_EPOCHS = 30          # un peu plus, mais avec early stop plus strict
EARLY_STOP_PATIENCE = 5  # arrête plus tôt si stagnation
MIN_LR = 1e-5             # plus bas si scheduler fort

# Learning
LEARNING_RATE = 1e-4       # un peu plus haut pour démarrer
WEIGHT_DECAY = 4e-3        # ↑ plus fort : empêche les poids de gonfler
SCHEDULER_PATIENCE = 10    # laisse le modèle respirer avant de baisser LR
SCHEDULER_FACTOR = 0.7     # réduction plus douce

# MixUp / Label smoothing
MIXUP_ALPHA = 0.4         # ↑ plus de bruit régulier
LABEL_SMOOTHING = 0.15     # ↑ adoucit les cibles

# Architecture
HIDDEN_DIM = 256          # ↓ moins de capacité
CLASSIFIER_DROPOUT = 0.5 # ↑ plus de dropout (meilleur contre l’overfit)

# Optionnel
GRAD_CLIP = 1.0           # empêche les gradients de diverger sur gros batchs


In [21]:
train_loader, _ = setup_multimodal_pipeline(
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    use_weighted_sampler=True,
    split="train",
)

val_loader, _ = setup_multimodal_pipeline(
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    use_weighted_sampler=False,
    split="validate",
)

[train] Indexed 6513 samples across 20 classes.
[validate] Indexed 497 samples across 20 classes.


In [ ]:
run_pipeline_unifier(train_loader, val_loader, NUM_CLASSES, save_path="best_unifier.pt")


--- Running Unifier Training Pipeline ---
Device: cuda
  LR: 0.0001, Weight Decay: 0.004
  MixUp α: 0.4, Dropout: 0.5
----------------------------------------

Epoch 1/30


Training:  19%|█▉        | 5/26 [00:06<00:24,  1.18s/it]

# Sauvegarde

# Evaluation